In [ ]:
# Dependencies are already installed in the local .venv (torch, timm, opencv-python, scikit-learn, av, psutil).
# Uncomment the line below only if running on a fresh environment:
# %pip install -q timm psutil opencv-python scikit-learn av

In [ ]:
import os
import gc
import json
import time
import traceback
from pathlib import Path

import cv2
import av  # Required for true hardware keyframe stream parsing
import numpy as np
import torch
import torch.nn as nn
import timm
import psutil

# ========================================================================================
# 1. GLOBAL CONFIGURATION (No static gop_size parameter)
# Paths are relative to this notebook's folder (system-codes/model).
# ========================================================================================
CONFIG = {
    "input_root": "dataset",
    "class_names": ["Violence", "NonViolence"],
    "output_root": "features",
    "video_extensions": (".mp4", ".avi", ".mov", ".mkv"),
    "frames_per_gop": 4,
    "neighborhood_gops": 3,
    "resize_dim": (380, 380),  # Native resolution for EfficientNet-B4
    "backbone_name": "efficientnet_b4",
    "error_log_path": "features/extraction_errors.json",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "feature_dim": 1792,
    "target_seq_len": 12,
}

# ========================================================================================
# 2. FROZEN BACKBONE MODEL INITIALIZATION (1,792 Channel Convolutional Head)
# ========================================================================================
def build_backbone(config: dict) -> nn.Module:
    """
    Builds a frozen EfficientNet-B4 feature extractor.
    Calling reset_classifier(num_classes=0) drops the final linear layers while keeping
    the 1792-channel 'conv_head' and global average pooling intact.
    """
    model = timm.create_model(config["backbone_name"], pretrained=True)
    model.reset_classifier(num_classes=0)
    model.eval()
    for param in model.parameters():
        param.requires_grad_(False)
    model.to(config["device"])
    return model

# ========================================================================================
# 3. MEMORY-SAFE DOWNAMPLED DECODING & NATURAL GOP CHUNKING (PyAV Keyframes)
# ========================================================================================
def decode_video_to_gop_chunks_safe(video_path: str, resize_dim: tuple):
    """
    Decodes video container packets using PyAV and applies immediate downsampling
    to 380x380 on the fly. This prevents massive raw pixel matrices from saturating RAM.
    """
    container = av.open(video_path)
    stream = container.streams.video[0]
    stream.thread_type = 'AUTO'

    chunks = []
    current_chunk_rgb = []
    current_chunk_gray = []
    total_frames = 0

    for frame in container.decode(video=0):
        # Decode packet directly to an RGB array
        img_rgb = frame.to_ndarray(format='rgb24')
        total_frames += 1

        # Downsample immediately to drop memory footprint by over 90% per frame
        img_resized = cv2.resize(img_rgb, resize_dim, interpolation=cv2.INTER_AREA)
        img_gray = cv2.cvtColor(img_resized, cv2.COLOR_RGB2GRAY)

        # Slice block organically when a true codec keyframe (I-frame) arrives
        if frame.key_frame and len(current_chunk_rgb) > 0:
            chunks.append({
                "rgb": current_chunk_rgb,
                "gray": current_chunk_gray
            })
            current_chunk_rgb = []
            current_chunk_gray = []

        current_chunk_rgb.append(img_resized)
        current_chunk_gray.append(img_gray)

    # Append trailing frames
    if len(current_chunk_rgb) > 0:
        chunks.append({
            "rgb": current_chunk_rgb,
            "gray": current_chunk_gray
        })

    container.close()
    if len(chunks) == 0:
        raise RuntimeError(f"Video container yielded 0 readable GOP structures: {video_path}")
    return chunks, total_frames

# ========================================================================================
# 4. UNIFORM RESAMPLING INDICES & LOW-FOOTPRINT MOTION SCORING
# ========================================================================================
def uniform_sample_indices(n_available: int, n_target: int):
    if n_available <= 0:
        raise RuntimeError("Cannot compute indices on an empty chunk.")
    if n_available >= n_target:
        return np.linspace(0, n_available - 1, n_target).round().astype(int)
    base = np.arange(n_available)
    reps = int(np.ceil(n_target / n_available))
    return np.tile(base, reps)[:n_target]

def compute_motion_score_safe(sampled_gray_frames):
    if len(sampled_gray_frames) < 2:
        return 0.0
    score = 0.0
    prev_gray = sampled_gray_frames[0]
    for curr_gray in sampled_gray_frames[1:]:
        diff = cv2.absdiff(curr_gray, prev_gray)
        score += float(np.sum(diff, dtype=np.float64))
        prev_gray = curr_gray
    return score

def select_neighborhood_chunk_indices(num_chunks: int, peak_idx: int, window: int = 3):
    if num_chunks < window:
        return None
    half = window // 2
    start = peak_idx - half
    end = start + window
    if start < 0:
        shift = -start
        start += shift
        end += shift
    if end > num_chunks:
        shift = end - num_chunks
        start -= shift
        end -= shift
    return list(range(max(0, start), min(num_chunks, end)))

def clone_frames_to_length(frames, target_len: int):
    if len(frames) == 0:
        raise RuntimeError("Cannot process an empty frame fallback matrix.")
    if len(frames) >= target_len:
        return frames[:target_len]
    reps = int(np.ceil(target_len / len(frames)))
    return (frames * reps)[:target_len]

# ========================================================================================
# 5. VECTORIZATION & MODEL INFERENCE
# ========================================================================================
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def frames_to_feature_tensor(frames, backbone_model, config):
    assert len(frames) == config["target_seq_len"]
    batch = np.stack(frames, axis=0)  # Stack uint8 arrays cheaply
    batch_t = torch.from_numpy(batch).permute(0, 3, 1, 2).to(config["device"], dtype=torch.float32)

    mean = IMAGENET_MEAN.to(config["device"])
    std = IMAGENET_STD.to(config["device"])
    batch_t = (batch_t / 255.0 - mean) / std

    with torch.no_grad():
        pooled = backbone_model(batch_t)  # Returns clean (12, 1792) embeddings directly

    result = pooled.detach().to("cpu")
    del batch_t, pooled
    return result

# ========================================================================================
# 6. SINGLE-VIDEO PIPELINE ORCHESTRATION (With Adaptive Short Video Interpolation)
# ========================================================================================
def process_single_video(video_path: str, out_path: str, backbone_model, config):
    # Call safe function with exactly one argument - no config["gop_size"] lookup!
    chunks, total_frames = decode_video_to_gop_chunks_safe(video_path, config["resize_dim"])
    num_chunks = len(chunks)

    # ADAPTIVE PLAN: If video is ultra-short, smoothly interpolate across the whole timeline
    if num_chunks < config["neighborhood_gops"]:
        all_rgb = []
        for c in chunks:
            all_rgb.extend(c["rgb"])
        adaptive_idxs = np.linspace(0, len(all_rgb) - 1, config["target_seq_len"]).round().astype(int)
        final_frames_rgb = [all_rgb[i] for i in adaptive_idxs]
        used_cloning = False
        peak_gop_idx = 0

    # STANDARD PLAN: Execute neighborhood extraction for healthy videos (>= 3 GOPs)
    else:
        motion_scores = []
        for c in chunks:
            idxs = uniform_sample_indices(len(c["gray"]), config["frames_per_gop"])
            sampled_gray = [c["gray"][i] for i in idxs]
            motion_scores.append(compute_motion_score_safe(sampled_gray))

        peak_gop_idx = int(np.argmax(motion_scores))
        used_cloning = False

        neighborhood_idxs = select_neighborhood_chunk_indices(
            num_chunks, peak_gop_idx, window=config["neighborhood_gops"]
        )

        final_frames_rgb = []
        for idx in neighborhood_idxs:
            target_chunk = chunks[idx]
            rep_idxs = uniform_sample_indices(len(target_chunk["rgb"]), config["frames_per_gop"])
            for r_i in rep_idxs:
                final_frames_rgb.append(target_chunk["rgb"][r_i])

    if len(final_frames_rgb) != config["target_seq_len"]:
        final_frames_rgb = clone_frames_to_length(final_frames_rgb, config["target_seq_len"])
        used_cloning = True

    # Run inference to generate the (12, 1792) feature layout
    feature_tensor = frames_to_feature_tensor(final_frames_rgb, backbone_model, config)
    assert feature_tensor.shape == (config["target_seq_len"], config["feature_dim"])
    torch.save(feature_tensor, out_path)

    meta = {
        "num_gops": num_chunks,
        "total_frames": total_frames,
        "peak_gop_idx": peak_gop_idx,
        "used_cloning_fallback": used_cloning,
        "output_shape": list(feature_tensor.shape),
    }

    # Deep explicit purging to prevent system RAM leaks
    del chunks, final_frames_rgb, feature_tensor
    gc.collect()
    if config["device"] == "cuda":
        torch.cuda.empty_cache()

    return meta

# ========================================================================================
# 7. DATASET DISCOVERY & EXECUTION DRIVER LOOP
# ========================================================================================
def discover_videos(config):
    video_list = []
    input_root = Path(config["input_root"])
    for class_name in config["class_names"]:
        class_dir = input_root / class_name
        if not class_dir.is_dir():
            print(f"WARNING: Class folder not found, skipping: {class_dir}")
            continue
        for ext in config["video_extensions"]:
            video_list.extend(sorted(class_dir.glob(f"*{ext}")))
            video_list.extend(sorted(class_dir.glob(f"*{ext.upper()}")))
    seen = set()
    paired = []
    for p in video_list:
        if p in seen:
            continue
        seen.add(p)
        paired.append((p.parent.name, p))
    return paired

def run_extraction(config, backbone_model):
    out_root = Path(config["output_root"])
    for class_name in config["class_names"]:
        (out_root / class_name).mkdir(parents=True, exist_ok=True)

    videos = discover_videos(config)
    total = len(videos)
    print(f"Discovered {total} candidate video files across {config['class_names']}.\n")

    errors = []
    success_count = 0
    skip_count = 0
    start_time = time.time()

    for i, (class_name, video_path) in enumerate(videos, start=1):
        video_id = video_path.stem
        out_path = out_root / class_name / f"{video_id}.pth"

        # Resumability framework
        if out_path.exists():
            skip_count += 1
            print(f"[{i}/{total}] {video_path.name:<45} | Status: SKIP (already exists)")
            continue

        try:
            meta = process_single_video(str(video_path), str(out_path), backbone_model, config)
            success_count += 1
            ram_pct = psutil.virtual_memory().percent
            print(
                f"[{i}/{total}] Video: {video_path.name:<40} | "
                f"Peak GOP: {meta['num_gops']:>3} Chunks | "
                f"Cloned: {str(meta['used_cloning_fallback']):<5} | "
                f"RAM: {ram_pct:>4.1f}% | Status: OK"
            )
        except Exception as exc:
            error_entry = {
                "video_path": str(video_path),
                "class_name": class_name,
                "error": str(exc),
                "traceback": traceback.format_exc(),
            }
            errors.append(error_entry)
            print(f"[{i}/{total}] Video: {video_path.name:<40} | Status: FAILED -> {exc}")
            gc.collect()
            if config["device"] == "cuda":
                torch.cuda.empty_cache()
            continue

    elapsed = time.time() - start_time
    with open(config["error_log_path"], "w") as f:
        json.dump(errors, f, indent=2)

    return {
        "total_discovered": total,
        "succeeded": success_count,
        "skipped_existing": skip_count,
        "failed": len(errors),
        "elapsed_seconds": elapsed,
    }

# ========================================================================================
# 8. EXECUTION PASSTHROUGH
# ========================================================================================
print("Loading frozen EfficientNet-B4 backbone...")
backbone = build_backbone(CONFIG)
print("Backbone ready. Executing extraction pass...")

summary = run_extraction(CONFIG, backbone)

print("\n" + "="*70)
print("EXTRACTION COMPLETE - FINAL REPORT")
print("="*70)
print(f"Total videos discovered : {summary['total_discovered']}")
print(f"Succeeded this run      : {summary['succeeded']}")
print(f"Skipped (already done)  : {summary['skipped_existing']}\n")
print(f"Failed / logged errors  : {summary['failed']}\n")
print(f"Elapsed time            : {summary['elapsed_seconds']:.1f}s")
print("="*70)